In [10]:
!pip install pyod
!pip install geopy

In [11]:
import json

with open('gt_macs.json', 'r') as file:
    gt_macs_data = json.load(file)

print("Ground Truth Loaded.")
print(gt_macs_data)

In [12]:
import json
from datetime import datetime, timedelta
from dateutil import parser
from itertools import pairwise
from geopy.distance import lonlat, distance
import pandas as pd
import numpy as np
from pyod.models.cblof import CBLOF
from sklearn.preprocessing import StandardScaler

DATETIME_FORMAT = '%a %b %d %X %Z %Y'
THRESHOLD_SECONDS = 600

def analyze_log_file(log_file_path: str, gt_macs_for_file: list):

    with open(log_file_path, 'r') as file:
        data = json.load(file)
    datas = data['detections']

    def from_json(json_data):
        return {
            'lat': json_data["lat"],
            'long': json_data["long"],
            'rssi': json_data["rssi"],
            'timestamp': parser.parse(json_data["t"])
        }

    trajectories = {}
    for detection in datas:
        if detection['mac'] not in trajectories:
            trajectories[detection['mac']] = []
        trajectories[detection['mac']].append(from_json(detection))

    mac_sum = []

    for k, v in trajectories.items():
        if len(v) <= 2:
            if len(v) == 1:
                mac_sum.append({
                    'mac': k,
                    'total_time': 0,
                    'total_dist': 0,
                    'valid_time': 0,
                    'valid_dist': 0,
                    'rssi': v[0]['rssi']
                })
            continue


        v = sorted(v, key=lambda x: x['timestamp'])


        t = [{
            'timestamp': y['timestamp'],
            'long': y['long'],
            'lat': y['lat'],
            'rssi': y['rssi']
        } for y in v]

        total_time = 0.0
        total_dist = 0.0

        pairs = list(pairwise(t))


        valid_pairs = []
        for p1, p2 in pairs:

            time_gap = p2['timestamp'] - p1['timestamp']
            time_gap_seconds = time_gap.total_seconds()

            current_dist = distance(
                lonlat(p1['long'], p1['lat']),
                lonlat(p2['long'], p2['lat'])
            ).m


            total_time += time_gap_seconds
            total_dist += current_dist


            if time_gap_seconds <= THRESHOLD_SECONDS:
                valid_pairs.append({
                    'start_time': p1['timestamp'],
                    'end_time': p2['timestamp'],
                    'duration': time_gap_seconds,
                    'distance': current_dist
                })

        if valid_pairs:
            segments = []
            current_segment = {
                'start_time': valid_pairs[0]['start_time'],
                'end_time': valid_pairs[0]['end_time'],
                'distance': valid_pairs[0]['distance']
            }

            for i in range(1, len(valid_pairs)):

                if valid_pairs[i]['start_time'] == current_segment['end_time']:
                    current_segment['end_time'] = valid_pairs[i]['end_time']
                    current_segment['distance'] += valid_pairs[i]['distance']
                else:

                    segments.append(current_segment)
                    current_segment = {
                        'start_time': valid_pairs[i]['start_time'],
                        'end_time': valid_pairs[i]['end_time'],
                        'distance': valid_pairs[i]['distance']
                    }


            segments.append(current_segment)

            valid_time = sum((seg['end_time'] - seg['start_time']).total_seconds()
                           for seg in segments)
            valid_dist = sum(seg['distance'] for seg in segments)
        else:
            valid_time = 0.0
            valid_dist = 0.0



        mean_rssi = np.mean([a['rssi'] for a in t])

        mac_sum.append({
            'mac': k,
            'total_time': total_time,
            'total_dist': total_dist,
            'valid_time': valid_time,
            'valid_dist': valid_dist,
            'rssi': mean_rssi
        })

    data_df = pd.DataFrame(mac_sum)

    if data_df.empty:
        return pd.DataFrame(), 0, 0, 0, 0


    features = ['valid_time','valid_dist','rssi']

    X_first = data_df[features]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_first)

    cblof = CBLOF(n_clusters=5, random_state=42)
    cblof.fit(X_scaled)

    data_df['cblof_scores'] = cblof.decision_scores_


    data_df_sorted = data_df.sort_values(by='cblof_scores', ascending=False).copy()


    data_df_sorted['score_diff'] = data_df_sorted['cblof_scores'] - data_df_sorted['cblof_scores'].shift(-1)
    # print(data_df_sorted)


    GAP_THRESHOLD = 2.4
    break_indices = np.where(data_df_sorted['score_diff'] >= GAP_THRESHOLD)[0]

    if break_indices.size > 0:
        break_point_index = break_indices[0]
        data_df_sorted['is_outlier'] = np.arange(len(data_df_sorted)) <= break_point_index
    else:

        data_df_sorted['is_outlier'] = False


    final_df = data_df.merge(data_df_sorted[['mac', 'is_outlier']], on='mac', how='left')


    final_df['ground_truth_outlier'] = final_df['mac'].apply(lambda x: 1 if x in gt_macs_for_file else 0)

    TP = ((final_df['is_outlier'] == True) & (final_df['ground_truth_outlier'] == 1)).sum()
    FP = ((final_df['is_outlier'] == True) & (final_df['ground_truth_outlier'] == 0)).sum()
    TN = ((final_df['is_outlier'] == False) & (final_df['ground_truth_outlier'] == 0)).sum()
    FN = ((final_df['is_outlier'] == False) & (final_df['ground_truth_outlier'] == 1)).sum()

    return final_df, TP, FP, TN, FN

In [13]:
all_results = []
total_TP = 0
total_FP = 0
total_TN = 0
total_FN = 0

for log_file_name, gt_macs_for_file in gt_macs_data.items():
    log_file_path = f"./{log_file_name}"

    try:

        df_result, TP, FP, TN, FN = analyze_log_file(log_file_path, gt_macs_for_file)

        all_results.append({
            'log_file': log_file_name,
            'data_frame': df_result,
            'TP': TP, 'FP': FP, 'TN': TN, 'FN': FN
        })

        total_TP += TP
        total_FP += FP
        total_TN += TN
        total_FN += FN

    except ValueError as e:
        if ("n_samples=" in str(e) and "should be >= n_clusters=" in str(e)) or \
           ("Could not form a valid cluster separation" in str(e)):
            print(f"Skipping analysis for {log_file_name} due to ValueError: {e}")
            all_results.append({
                'log_file': log_file_name,
                'data_frame': pd.DataFrame(),
                'TP': 0, 'FP': 0, 'TN': 0, 'FN': 0
            })
        else:
            raise

print("Analysis for all files completed successfully.")
print(f"Total True Positives (TP): {total_TP}")
print(f"Total False Positives (FP): {total_FP}")
print(f"Total True Negatives (TN): {total_TN}")
print(f"Total False Negatives (FN): {total_FN}")

In [14]:
import warnings

# precision
if (total_TP + total_FP) > 0:
    precision = total_TP / (total_TP + total_FP)
else:
    precision = 0.0

# recall
if (total_TP + total_FN) > 0:
    recall = total_TP / (total_TP + total_FN)
else:
    recall = 0.0

# f1-score
if (precision + recall) > 0:
    f1_score = 2 * (precision * recall) / (precision + recall)
else:
    f1_score = 0.0

print(f"\nOverall Precision: {precision:.4f}")
print(f"Overall Recall: {recall:.4f}")
print(f"Overall F1-Score: {f1_score:.4f}")

In [15]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


confusion_matrix_data = np.array([[total_TP, total_FP], [total_FN, total_TN]])

labels = np.array([['TP', 'FP'], ['FN', 'TN']])
annotations = np.array([["TP\n" + str(total_TP), "FP\n" + str(total_FP)],
                      ["FN\n" + str(total_FN), "TN\n" + str(total_TN)]])

plt.figure(figsize=(8, 6))
sns.heatmap(confusion_matrix_data, annot=annotations, fmt='', cmap='OrRd', cbar=False)


# plt.title('Aggregate Confusion Matrix for CBLOF Detection')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


In [16]:
print("\n ***Individual Log File Results ***")
for result in all_results:
    print(f"\nLog File: {result['log_file']}")
    print(f"  True Positives (TP): {result['TP']}")
    print(f"  False Positives (FP): {result['FP']}")
    print(f"  True Negatives (TN): {result['TN']}")
    print(f"  False Negatives (FN): {result['FN']}")

    precision_file = result['TP'] / (result['TP'] + result['FP']) if (result['TP'] + result['FP']) > 0 else 0.0
    recall_file = result['TP'] / (result['TP'] + result['FN']) if (result['TP'] + result['FN']) > 0 else 0.0
    f1_score_file = 2 * (precision_file * recall_file) / (precision_file + recall_file) if (precision_file + recall_file) > 0 else 0.0

    print(f"  Precision: {precision_file:.4f}")
    print(f"  Recall: {recall_file:.4f}")
    print(f"  F1-Score: {f1_score_file:.4f}")

##Analysis over time

In [17]:
import matplotlib.pyplot as plt

def analyze_log_file_over_time(log_file_path: str, gt_macs_for_file: list):

    with open(log_file_path, 'r') as file:
        data = json.load(file)

    detections = data['detections']

    for d in detections:
        d['timestamp'] = parser.parse(d["t"])

    start_time = min(d['timestamp'] for d in detections)
    end_time = max(d['timestamp'] for d in detections)

    total_minutes = int((end_time - start_time).total_seconds() // 60) + 1

    results = []

    print("\n****DETECTIONS PER MINUTE****\n")


    for minute in range(1, total_minutes + 1):

        cutoff = start_time + timedelta(minutes=minute)

        subset = [
            d for d in detections
            if d['timestamp'] <= cutoff
        ]

        if len(subset) == 0:
            continue

        trajectories = {}
        for detection in subset:
            if detection['mac'] not in trajectories:
                trajectories[detection['mac']] = []
            trajectories[detection['mac']].append({
                'lat': detection["lat"],
                'long': detection["long"],
                'rssi': detection["rssi"],
                'timestamp': detection["timestamp"]
            })

        mac_sum = []

        for k, v in trajectories.items():

            if len(v) <= 2:
                continue

            v = sorted(v, key=lambda x: x['timestamp'])

            t = [{
                'timestamp': y['timestamp'],
                'long': y['long'],
                'lat': y['lat'],
                'rssi': y['rssi']
            } for y in v]

            total_time = 0.0
            total_dist = 0.0

            pairs = list(pairwise(t))
            valid_pairs = []

            for p1, p2 in pairs:

                time_gap_seconds = (p2['timestamp'] - p1['timestamp']).total_seconds()

                current_dist = distance(
                    lonlat(p1['long'], p1['lat']),
                    lonlat(p2['long'], p2['lat'])
                ).m

                total_time += time_gap_seconds
                total_dist += current_dist

                if time_gap_seconds <= THRESHOLD_SECONDS:
                    valid_pairs.append({
                        'start_time': p1['timestamp'],
                        'end_time': p2['timestamp'],
                        'distance': current_dist
                    })

            if valid_pairs:
                segments = []
                current_segment = valid_pairs[0]

                for i in range(1, len(valid_pairs)):
                    if valid_pairs[i]['start_time'] == current_segment['end_time']:
                        current_segment['end_time'] = valid_pairs[i]['end_time']
                        current_segment['distance'] += valid_pairs[i]['distance']
                    else:
                        segments.append(current_segment)
                        current_segment = valid_pairs[i]

                segments.append(current_segment)

                valid_time = sum(
                    (seg['end_time'] - seg['start_time']).total_seconds()
                    for seg in segments
                )
                valid_dist = sum(seg['distance'] for seg in segments)
            else:
                valid_time = 0.0
                valid_dist = 0.0

            mean_rssi = np.mean([a['rssi'] for a in t])

            mac_sum.append({
                'mac': k,
                'valid_time': valid_time,
                'valid_dist': valid_dist,
                'rssi': mean_rssi
            })

        data_df = pd.DataFrame(mac_sum)

        if len(data_df) < 5:
            continue
        features = ['valid_time', 'valid_dist', 'rssi']

        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(data_df[features])

        # n_clusters = min(5, len(data_df))
        # cblof = CBLOF(n_clusters=n_clusters, random_state=42)
        # cblof.fit(X_scaled)

        # data_df['cblof_scores'] = cblof.decision_scores_
        n_clusters = min(5, len(data_df))
        cblof = CBLOF(n_clusters=n_clusters, random_state=42)

        try:
            cblof.fit(X_scaled)
        except Exception as e:
            print(f"Minute {minute:2d} | skip due to clustering error: {e}")
            continue

        data_df['cblof_scores'] = cblof.decision_scores_

        data_df_sorted = data_df.sort_values(by='cblof_scores', ascending=False).copy()
        data_df_sorted['score_diff'] = (
            data_df_sorted['cblof_scores']
            - data_df_sorted['cblof_scores'].shift(-1)
        )

        GAP_THRESHOLD = 2.4
        break_indices = np.where(data_df_sorted['score_diff'] >= GAP_THRESHOLD)[0]

        if break_indices.size > 0:
            break_point_index = break_indices[0]
            data_df_sorted['is_outlier'] = (
                np.arange(len(data_df_sorted)) <= break_point_index
            )
        else:
            data_df_sorted['is_outlier'] = False

        final_df = data_df.merge(
            data_df_sorted[['mac', 'is_outlier']],
            on='mac',
            how='left'
        )

        final_df['ground_truth_outlier'] = final_df['mac'].apply(
            lambda x: 1 if x in gt_macs_for_file else 0
        )

        TP = ((final_df['is_outlier']) & (final_df['ground_truth_outlier'] == 1)).sum()
        FP = ((final_df['is_outlier']) & (final_df['ground_truth_outlier'] == 0)).sum()
        TN = ((~final_df['is_outlier']) & (final_df['ground_truth_outlier'] == 0)).sum()
        FN = ((~final_df['is_outlier']) & (final_df['ground_truth_outlier'] == 1)).sum()

        if TP + FP == 0 or TP + FN == 0:
            F1 = 0
        else:
            precision = TP / (TP + FP)
            recall = TP / (TP + FN)
            F1 = 0 if (precision + recall) == 0 else 2 * precision * recall / (precision + recall)

        print(f"Minute {minute:2d} || TP={TP} FP={FP} TN={TN} FN={FN} F1={F1:.3f}")

        results.append({
            "minute": minute,
            "TP": TP,
            "FP": FP,
            "TN": TN,
            "FN": FN,
            "F1": F1
        })

    results_df = pd.DataFrame(results)

    if not results_df.empty:

        plt.figure()
        plt.plot(results_df["minute"], results_df["F1"])
        plt.xlabel("Minutes",fontsize=23,fontweight='bold')
        plt.ylabel("F1 Score", fontsize=23,fontweight='bold')
        plt.tick_params(axis='both', labelsize=25)
        for label in plt.gca().get_xticklabels():
            label.set_fontweight('bold')
        for label in plt.gca().get_yticklabels():
            label.set_fontweight('bold')
        plt.show()

        # plt.figure()
        # plt.plot(results_df["minute"], results_df["TP"])
        # plt.xlabel("Minute")
        # plt.ylabel("TP")
        # plt.title("TP Over Time")
        # plt.show()

        # plt.figure()
        # plt.plot(results_df["minute"], results_df["FP"])
        # plt.xlabel("Minute")
        # plt.ylabel("FP")
        # plt.title("FP Over Time")
        # plt.show()

    return results_df


In [18]:
all_over_time_results = {}

for log_file_name, gt_macs_for_file in gt_macs_data.items():

    log_file_path = f"./{log_file_name}"

    print(f"printing: {log_file_name}")

    try:
        results_df = analyze_log_file_over_time(
            log_file_path,
            gt_macs_for_file
        )

        all_over_time_results[log_file_name] = results_df

    except ValueError as e:

        if ("n_samples=" in str(e) and "should be >= n_clusters=" in str(e)) or \
           ("Could not form a valid cluster separation" in str(e)):

            print(f"Skipping {log_file_name} due to clustering error: {e}")
            all_over_time_results[log_file_name] = pd.DataFrame()

        else:
            raise

print("\nDone")
